## End-to-End Demo

Single-track inference through the full Task 3 pipeline:

`audio -> segment graph -> GraphSAGE embedding` and `artist bio -> DistilBERT embedding`,
fused via the trained **early-concat** model to predict contextual tags (genre +
contextual/music tags, including mood-related ones).

Uses the completed run at `results/task3/20260907-015512/` (all 4 ablation variants trained
on the fixed graph builder + combined genre+tag label vocab; early-concat is the best
variant on all 3 metrics: test macro_f1=0.202, micro_f1=0.246, auc_pr=0.287 — see that
run's `metrics.json` for the full ablation comparison).

> **Note:** `early_concat_best_model.pt` (~250MB) is **not committed to git** (`*.pt` is
> gitignored — checkpoints are too large for a plain repo). This notebook's saved outputs
> below were produced from that checkpoint on the machine it was trained on. To re-run it
> yourself, first train Task 3 (`python -m src.train --task 3 --freeze-layers 2`, after
> `python -m src.train --task 1 --freeze-layers 2` and `python -m src.musiccaps_prep
> --fetch-metadata --download-audio`), then point `RUN_DIR` below at your new run
> directory — see the README's "Reproducing the demo notebook" section.


In [ ]:
import json
import sys
from pathlib import Path

import torch
from transformers import AutoTokenizer

sys.path.insert(0, str(Path.cwd().parent))

from src.datasets import (
    build_task1_dataset,
    load_corrupted_track_ids,
    load_fma_metadata,
    load_fma_splits,
)
from src.fusion_model import FusionModel
from src.graph_builder import build_or_load_track_graph
from src.utils import load_config

ROOT = Path.cwd().parent
config = load_config(ROOT / "config.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"

RUN_DIR = ROOT / "results/task3/20260907-015512"
with (RUN_DIR / "metrics.json").open() as f:
    task3_metrics = json.load(f)
top_tags = task3_metrics["top_tags"]  # 20 plain tags, used only to build the demo track's true-tag display
label_names = task3_metrics["label_names"]  # 36 genre+tag labels, the model's actual output vocabulary
thresholds = task3_metrics["ablations"]["early_concat"]["per_tag_thresholds"]
num_labels = len(label_names)

print(f"device={device}")
print(f"loaded run: {RUN_DIR.name}")
print(f"early_concat test metrics: {task3_metrics['ablations']['early_concat']['test']}")


device=cuda
loaded run: 20260907-015512
early_concat test metrics: {'macro_f1': 0.20223479724517476, 'micro_f1': 0.24634420697412823, 'auc_pr': 0.28671667562473724, 'loss': 0.35246633287845003}


## Pick one test-set track

Uses the same Task 1 dataset construction as training: the input text is the raw artist bio.


In [2]:
tracks = load_fma_metadata(ROOT / "data/raw/fma_metadata", subset="medium")
corrupted_ids = load_corrupted_track_ids()
splits = load_fma_splits(tracks, exclude_track_ids=corrupted_ids)
task_df = build_task1_dataset(tracks, top_tags)
split_of = {tid: name for name, ids in splits.items() for tid in ids}
task_df = task_df.assign(split=task_df["track_id"].map(split_of))
test_df = task_df[task_df["split"] == "test"].reset_index(drop=True)

row = test_df[test_df["labels"].apply(sum) > 0].iloc[0]  # pick one with >=1 true tag, for a clearer demo
track_id = int(row["track_id"])
genre = tracks.loc[tracks["track_id"] == track_id, "genre_top"].item()
true_tags = [top_tags[i] for i, v in enumerate(row["labels"]) if v == 1]

print(f"track_id={track_id}  genre_top={genre}")
print(f"bio text: {row['text'][:300]}")
print(f"true contextual tags: {true_tags}")


track_id=13814  genre_top=Folk
bio text: Menhirs of Er Grah (Tom Carter) is a folk band named after some prehistoric standing-stones in France. 
 Tom Carter lives in London, and has released a number of lo-fi/electronica albums under the anagram of March Rosetta.
true contextual tags: ['acoustic', 'clinical archives']


## Audio -> segment graph

In [3]:
audio_cfg = config["audio"]
graph_cfg = config["graph"]
in_dim = 2 * audio_cfg["n_mfcc"] + (24 if audio_cfg["use_chroma"] else 0)

graph = build_or_load_track_graph(
    track_id=track_id,
    audio_root=config["dataset"]["root"],
    cache_dir=ROOT / "data/processed/graph_cache",
    sample_rate=config["dataset"]["sample_rate"],
    segment_seconds=audio_cfg["segment_seconds"],
    n_mfcc=audio_cfg["n_mfcc"],
    use_chroma=audio_cfg["use_chroma"],
    similarity_threshold=graph_cfg["similarity_threshold"],
    bidirectional=graph_cfg["bidirectional_edges"],
    self_loops=graph_cfg["self_loops"],
)
print(f"segment graph for track {track_id}: {graph.x.shape[0]} nodes (5s segments), "
      f"{graph.edge_index.shape[1]} directed edges, node feature dim={graph.x.shape[1]}")

segment graph for track 13814: 6 nodes (5s segments), 10 directed edges, node feature dim=64


## Text -> BERT, fusion -> tag prediction

Loads the trained `early_concat` fusion checkpoint and runs a single forward pass, then
applies the per-tag thresholds tuned on the validation set (same ones used at test time).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config["bert"]["model_name"])
encoded = tokenizer(
    row["text"], truncation=True, max_length=config["bert"]["max_length"],
    padding="max_length", return_tensors="pt",
)

model = FusionModel(
    graph_in_dim=in_dim,
    graph_hidden_dim=config["gnn"]["hidden_dim"],
    graph_num_layers=config["gnn"]["num_layers"],
    graph_dropout=config["gnn"]["dropout"],
    bert_model_name=config["bert"]["model_name"],
    bert_freeze_layers=config["bert"]["freeze_layers"],
    num_labels=num_labels,
    mode="early_concat",
).to(device)
model.load_state_dict(torch.load(RUN_DIR / "early_concat_best_model.pt", map_location=device))
model.eval()

batch_index = torch.zeros(graph.x.shape[0], dtype=torch.long)  # single graph -> single PyG "batch"
with torch.no_grad():
    logits = model(
        graph.x.to(device), graph.edge_index.to(device), batch_index.to(device),
        encoded["input_ids"].to(device), encoded["attention_mask"].to(device),
    )
    probs = torch.sigmoid(logits)[0].cpu().numpy()

predicted_labels = [label_names[i] for i, p in enumerate(probs) if p >= thresholds[i]]
print(f"predicted contextual labels (early-concat fusion, genre+tag): {predicted_labels}")
print(f"true contextual tags:                                          {true_tags}")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4654.91it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


predicted contextual labels (early-concat fusion, genre+mood): ['genre:Electronic', 'genre:Pop', 'electronic', 'electronica', 'electro']
true contextual tags:                                          ['acoustic', 'clinical archives']
